In [ ]:
# imports

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# paths and plotting style

PROCESSED_DIR = Path("processed_data")
FINAL_RESULTS_DIR = Path("results") / "final_models"
CAPACITY_RESULTS_DIR = Path("results") / "capacity"
WAKE_RESULTS_DIR = Path("results") / "wake_analysis"
FIGURE_DIR = Path("figures") / "results"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SCADA_FILE = PROCESSED_DIR / "penmanshiel_scada_complete.parquet"
STATIC_FILE = PROCESSED_DIR / "penmanshiel_static.parquet"

SEEDS = [1, 21, 42, 84, 123]
REFERENCE_SEED = 42

COLOR_A = "#007191"
COLOR_B = "orange"
COLOR_C = "#7A7A7A"
COLOR_OBSERVED = "black"
COLOR_REFERENCE = "steelblue"

WAKE_COLORS = {
    "Low": "#D4DFEA",
    "Medium": "#829AB5",
    "High": "#3E5871",
}

LABEL_FONTSIZE = 14
TICK_FONTSIZE = 12
LEGEND_FONTSIZE = 10


def format_axes(ax, grid_axis="both"):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

    ax.tick_params(
        axis="both",
        direction="out",
        length=5,
        width=1,
        labelsize=TICK_FONTSIZE,
        top=False,
        right=False,
    )

    ax.grid(
        axis=grid_axis,
        alpha=0.20,
        linewidth=0.8,
    )

    ax.set_axisbelow(True)


In [ ]:
# load final model outputs

seed_summary = pd.read_csv(
    FINAL_RESULTS_DIR / "seed_summary.csv"
)

reference_metrics = pd.read_csv(
    FINAL_RESULTS_DIR / "reference_metrics.csv"
)

predictions = {
    seed: pd.read_parquet(
        FINAL_RESULTS_DIR
        / f"predictions_seed_{seed}.parquet"
    )
    for seed in SEEDS
}

for seed in SEEDS:
    predictions[seed].index = pd.to_datetime(
        predictions[seed].index,
        utc=True,
    )

reference_predictions = predictions[SEEDS[0]]

for seed in SEEDS[1:]:
    if not reference_predictions.index.equals(
        predictions[seed].index
    ):
        raise ValueError(
            f"test timestamps do not align for seed {seed}"
        )

print(
    f"test timestamps: "
    f"{len(reference_predictions):,}"
)


In [ ]:
# table 5.1 farm-level predictive performance

metric_rows = []

linear = reference_metrics.loc[
    reference_metrics["reference"]
    == "Linear regression"
].iloc[0]

metric_rows.append(
    {
        "Model": "Linear regression",
        "Farm MAE (kW)": linear["MAE_kW"],
        "Farm RMSE (kW)": linear["RMSE_kW"],
        "Farm Bias (kW)": np.nan,
    }
)

for model in ["A", "B", "C"]:
    metric_rows.append(
        {
            "Model": f"Model {model}",
            "Farm MAE (kW)": (
                seed_summary[f"{model}_MAE_kW"].mean()
            ),
            "Farm MAE SD (kW)": (
                seed_summary[f"{model}_MAE_kW"].std(ddof=1)
            ),
            "Farm RMSE (kW)": (
                seed_summary[f"{model}_RMSE_kW"].mean()
            ),
            "Farm RMSE SD (kW)": (
                seed_summary[f"{model}_RMSE_kW"].std(ddof=1)
            ),
            "Farm Bias (kW)": (
                seed_summary[f"{model}_bias_kW"].mean()
            ),
            "Farm Bias SD (kW)": (
                seed_summary[f"{model}_bias_kW"].std(ddof=1)
            ),
        }
    )

table_5_1 = pd.DataFrame(metric_rows)

print(table_5_1.round(1).to_string(index=False))


In [ ]:
# five-seed mean predictions used in figures 5.1 and 5.2

plot_data = pd.DataFrame(
    index=reference_predictions.index
)

plot_data["global_ws"] = (
    reference_predictions["global_ws"]
)

plot_data["actual_power"] = (
    reference_predictions["actual_farm_power"]
)

for model in ["A", "B", "C"]:
    stacked_predictions = np.stack(
        [
            predictions[seed][
                f"pred_{model}_power"
            ].to_numpy()
            for seed in SEEDS
        ],
        axis=0,
    )

    plot_data[f"pred_{model}_power"] = (
        stacked_predictions.mean(axis=0)
    )


In [ ]:
# figure 5.1 observed and predicted farm power

point_size = 3
point_alpha = 0.12

fig, axes = plt.subplots(
    2,
    1,
    figsize=(9, 10),
    sharex=True,
)

axes[0].scatter(
    plot_data["global_ws"],
    plot_data["actual_power"],
    s=point_size,
    alpha=point_alpha,
    color=COLOR_OBSERVED,
    label="Observed",
)

axes[0].scatter(
    plot_data["global_ws"],
    plot_data["pred_A_power"],
    s=point_size,
    alpha=0.20,
    color=COLOR_A,
    label="Model A",
)

axes[0].scatter(
    plot_data["global_ws"],
    plot_data["pred_B_power"],
    s=point_size,
    alpha=0.20,
    color=COLOR_B,
    label="Model B",
)

axes[1].scatter(
    plot_data["global_ws"],
    plot_data["actual_power"],
    s=point_size,
    alpha=point_alpha,
    color=COLOR_OBSERVED,
    label="Observed",
)

axes[1].scatter(
    plot_data["global_ws"],
    plot_data["pred_A_power"],
    s=point_size,
    alpha=0.20,
    color=COLOR_A,
    label="Model A",
)

axes[1].scatter(
    plot_data["global_ws"],
    plot_data["pred_C_power"],
    s=point_size,
    alpha=0.20,
    color=COLOR_C,
    label="Model C",
)

for ax in axes:
    ax.set_ylabel(
        "Total farm power (kW)",
        fontsize=LABEL_FONTSIZE,
    )

    ax.set_ylim(0, 30000)
    format_axes(ax)

    ax.legend(
        fontsize=LEGEND_FONTSIZE,
        markerscale=3,
        frameon=False,
    )

axes[1].set_xlabel(
    "Global wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xlim(3.5, 25)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_1_farm_power_predictions.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 5.2 predicted farm-power spread relative to observed spread

wind_speed_edges = np.arange(
    3.5,
    20.5,
    1.0,
)

wind_speed_labels = np.arange(
    4,
    20,
)

spread_data = plot_data.copy()

spread_data["ws_bin"] = pd.cut(
    spread_data["global_ws"],
    bins=wind_speed_edges,
    labels=wind_speed_labels,
    right=False,
)

def p90_range(values):
    return (
        values.quantile(0.95)
        - values.quantile(0.05)
    )

spread = (
    spread_data
    .dropna(subset=["ws_bin"])
    .groupby(
        "ws_bin",
        observed=True,
    )
    .agg(
        n=("actual_power", "size"),
        observed_spread=("actual_power", p90_range),
        A_spread=("pred_A_power", p90_range),
        B_spread=("pred_B_power", p90_range),
        C_spread=("pred_C_power", p90_range),
    )
    .reset_index()
)

spread["wind_speed"] = (
    spread["ws_bin"].astype(float)
)

for model in ["A", "B", "C"]:
    spread[f"{model}_pct_observed"] = (
        100
        * spread[f"{model}_spread"]
        / spread["observed_spread"]
    )

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    spread["wind_speed"],
    spread["A_pct_observed"],
    marker="o",
    linewidth=2,
    color=COLOR_A,
    label="Model A",
)

ax.plot(
    spread["wind_speed"],
    spread["B_pct_observed"],
    marker="o",
    linewidth=2,
    color=COLOR_B,
    label="Model B",
)

ax.plot(
    spread["wind_speed"],
    spread["C_pct_observed"],
    marker="o",
    linewidth=2,
    color=COLOR_C,
    label="Model C",
)

ax.axhline(
    100,
    color="black",
    linestyle="--",
    linewidth=1.4,
    label="Observed spread",
)

ax.set_xlabel(
    "Global wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Predicted farm-power spread (% of observed)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_xlim(4, 19)

format_axes(ax)

ax.legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_2_farm_power_spread.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    spread[
        [
            "wind_speed",
            "n",
            "A_pct_observed",
            "B_pct_observed",
            "C_pct_observed",
        ]
    ]
    .round(1)
    .to_string(index=False)
)


In [ ]:
# table 5.2 network-capacity results

capacity_results = pd.read_csv(
    CAPACITY_RESULTS_DIR / "capacity_results.csv"
)

table_5_2 = capacity_results[
    [
        "hidden_layers",
        "A_train_kW",
        "A_validation_kW",
        "B_train_ms",
        "B_validation_ms",
        "C_train_kW",
        "C_validation_kW",
    ]
].copy()

print(
    table_5_2
    .round(3)
    .to_string(index=False)
)


In [ ]:
# figure 5.3 training and validation histories for seed 42

history = pd.read_csv(
    FINAL_RESULTS_DIR
    / f"loss_history_seed_{REFERENCE_SEED}.csv"
)

epoch = np.arange(
    1,
    len(history) + 1,
)

best_epochs = {}

for model in ["A", "B", "C"]:
    validation_column = (
        f"{model}_validation"
    )

    valid = history[
        validation_column
    ].dropna()

    best_index = valid.idxmin()
    best_epochs[model] = int(best_index + 1)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 3.8),
)

for ax, model in zip(
    axes,
    ["A", "B", "C"],
):
    train_values = history[
        f"{model}_train"
    ]

    validation_values = history[
        f"{model}_validation"
    ]

    ax.plot(
        epoch,
        train_values,
        label="Training",
    )

    ax.plot(
        epoch,
        validation_values,
        label="Validation",
    )

    ax.axvline(
        best_epochs[model],
        linestyle="--",
        linewidth=1,
        color=COLOR_A,
        alpha=0.7,
    )

    ax.set_title(
        f"Model {model}",
        fontsize=12,
    )

    ax.set_xlabel(
        "Epoch",
        fontsize=11,
    )

    ax.set_ylabel(
        "Huber loss",
        fontsize=11,
    )

    format_axes(ax)

axes[0].legend(
    frameon=False,
    fontsize=9,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_3_training_validation_histories.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

for model in ["A", "B", "C"]:
    value = history.loc[
        best_epochs[model] - 1,
        f"{model}_validation",
    ]

    print(
        f"Model {model}: "
        f"epoch {best_epochs[model]}, "
        f"validation loss {value:.6f}"
    )


In [ ]:
# optional seed-stability figure

fig, ax = plt.subplots(
    figsize=(6.5, 4.8)
)

models = ["A", "B", "C"]
x = np.arange(len(models))
offsets = np.linspace(
    -0.07,
    0.07,
    len(SEEDS),
)

means = []
stds = []

for model_index, model in enumerate(models):
    values = seed_summary[
        f"{model}_MAE_kW"
    ].to_numpy()

    ax.scatter(
        x[model_index] + offsets,
        values,
        s=40,
        color="0.55",
        alpha=0.8,
        zorder=2,
    )

    means.append(values.mean())
    stds.append(values.std(ddof=1))

ax.errorbar(
    x,
    means,
    yerr=stds,
    fmt="D",
    markersize=7,
    color="black",
    ecolor="black",
    elinewidth=1.4,
    capsize=4,
    zorder=4,
)

ax.set_xticks(x)
ax.set_xticklabels(
    ["Model A", "Model B", "Model C"]
)

ax.set_ylabel(
    "Farm-power MAE (kW)",
    fontsize=LABEL_FONTSIZE,
)

format_axes(
    ax,
    grid_axis="y",
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "farm_power_MAE_seed_stability.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 5.4 per-turbine predictive performance

turbine_ids = sorted(
    int(
        column.replace(
            "actual_ws_t",
            "",
        )
    )
    for column in reference_predictions.columns
    if column.startswith("actual_ws_t")
)

B_seed_rows = []
C_seed_rows = []

for seed in SEEDS:
    data = predictions[seed]

    for turbine_id in turbine_ids:
        B_seed_rows.append(
            {
                "seed": seed,
                "turbine_id": turbine_id,
                "MAE": np.mean(
                    np.abs(
                        data[
                            f"pred_B_ws_t{turbine_id}"
                        ]
                        - data[
                            f"actual_ws_t{turbine_id}"
                        ]
                    )
                ),
            }
        )

        C_seed_rows.append(
            {
                "seed": seed,
                "turbine_id": turbine_id,
                "MAE": np.mean(
                    np.abs(
                        data[
                            f"pred_C_power_t{turbine_id}"
                        ]
                        - data[
                            f"actual_power_t{turbine_id}"
                        ]
                    )
                ),
            }
        )

B_turbine = pd.DataFrame(
    B_seed_rows
).groupby("turbine_id").agg(
    mean=("MAE", "mean"),
    sd=("MAE", "std"),
)

C_turbine = pd.DataFrame(
    C_seed_rows
).groupby("turbine_id").agg(
    mean=("MAE", "mean"),
    sd=("MAE", "std"),
)

labels = [
    f"T{turbine_id:02d}"
    for turbine_id in turbine_ids
]

x = np.arange(len(turbine_ids))

fig, axes = plt.subplots(
    2,
    1,
    figsize=(8.2, 7.2),
    sharex=True,
)

axes[0].bar(
    x,
    B_turbine.loc[
        turbine_ids,
        "mean",
    ],
    yerr=B_turbine.loc[
        turbine_ids,
        "sd",
    ],
    capsize=2.5,
    color=COLOR_B,
)

axes[0].set_ylabel(
    "Wind-speed MAE (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].bar(
    x,
    C_turbine.loc[
        turbine_ids,
        "mean",
    ],
    yerr=C_turbine.loc[
        turbine_ids,
        "sd",
    ],
    capsize=2.5,
    color=COLOR_C,
)

axes[1].set_ylabel(
    "Power MAE (kW)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xlabel(
    "Turbine",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)

for ax in axes:
    format_axes(
        ax,
        grid_axis="y",
    )

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_4_per_turbine_error.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# table 5.3 Model B and measured-speed power-curve reference

B_mae_mean = seed_summary["B_MAE_kW"].mean()
B_mae_sd = seed_summary["B_MAE_kW"].std(ddof=1)

B_rmse_mean = seed_summary["B_RMSE_kW"].mean()
B_rmse_sd = seed_summary["B_RMSE_kW"].std(ddof=1)

B_bias_mean = seed_summary["B_bias_kW"].mean()
B_bias_sd = seed_summary["B_bias_kW"].std(ddof=1)

measured_reference = reference_metrics.loc[
    reference_metrics["reference"]
    == "Measured-speed reference"
].iloc[0]

table_5_3 = pd.DataFrame(
    [
        {
            "Reference": "Model B",
            "MAE mean (kW)": B_mae_mean,
            "MAE SD (kW)": B_mae_sd,
            "RMSE mean (kW)": B_rmse_mean,
            "RMSE SD (kW)": B_rmse_sd,
            "Bias mean (kW)": B_bias_mean,
            "Bias SD (kW)": B_bias_sd,
        },
        {
            "Reference": "Measured-speed reference",
            "MAE mean (kW)": measured_reference["MAE_kW"],
            "RMSE mean (kW)": measured_reference["RMSE_kW"],
            "Bias mean (kW)": np.nan,
        },
    ]
)

print(
    table_5_3
    .round(1)
    .to_string(index=False)
)


In [ ]:
# figure 5.5 Model B error decomposition by wind-speed bin

bin_edges = np.arange(
    3.5,
    20.5,
    1.0,
)

bin_labels = [
    f"{value}-{value + 1}"
    for value in range(4, 20)
]

seed_B_bin_rows = []

for seed in SEEDS:
    data = predictions[seed].copy()

    data["ws_bin"] = pd.cut(
        data["global_ws"],
        bins=bin_edges,
        labels=bin_labels,
        right=False,
    )

    data["B_abs_error"] = np.abs(
        data["pred_B_power"]
        - data["actual_farm_power"]
    )

    data["oracle_abs_error"] = np.abs(
        data["oracle_power"]
        - data["actual_farm_power"]
    )

    B_ws_errors = np.column_stack(
        [
            np.abs(
                data[f"pred_B_ws_t{turbine_id}"]
                - data[
                    f"actual_ws_t{turbine_id}"
                ]
            )
            for turbine_id in turbine_ids
        ]
    )

    data["B_ws_MAE"] = (
        B_ws_errors.mean(axis=1)
    )

    grouped = (
        data
        .dropna(subset=["ws_bin"])
        .groupby(
            "ws_bin",
            observed=True,
        )
        .agg(
            B_farm_MAE=(
                "B_abs_error",
                "mean",
            ),
            oracle_farm_MAE=(
                "oracle_abs_error",
                "mean",
            ),
            B_ws_MAE=(
                "B_ws_MAE",
                "mean",
            ),
        )
        .reset_index()
    )

    grouped["seed"] = seed
    seed_B_bin_rows.append(grouped)

B_error_by_bin = pd.concat(
    seed_B_bin_rows,
    ignore_index=True,
)

B_error_summary = (
    B_error_by_bin
    .groupby(
        "ws_bin",
        observed=True,
    )
    .agg(
        B_farm_MAE=(
            "B_farm_MAE",
            "mean",
        ),
        oracle_farm_MAE=(
            "oracle_farm_MAE",
            "mean",
        ),
        B_ws_MAE=(
            "B_ws_MAE",
            "mean",
        ),
    )
    .reindex(bin_labels)
    .reset_index()
)

x = np.arange(len(B_error_summary))

fig, axes = plt.subplots(
    2,
    1,
    figsize=(8.5, 7.5),
    sharex=True,
)

axes[0].plot(
    x,
    B_error_summary["B_farm_MAE"],
    marker="o",
    linewidth=2,
    color=COLOR_B,
    label="Model B",
)

axes[0].plot(
    x,
    B_error_summary["oracle_farm_MAE"],
    marker="o",
    linewidth=2,
    color=COLOR_REFERENCE,
    label="Measured-speed reference",
)

axes[0].set_ylabel(
    "Farm-power MAE (kW)",
    fontsize=LABEL_FONTSIZE,
)

axes[0].legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

axes[1].plot(
    x,
    B_error_summary["B_ws_MAE"],
    marker="o",
    linewidth=2,
    color=COLOR_B,
)

axes[1].set_ylabel(
    "Turbine wind-speed MAE (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xlabel(
    "Global wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xticks(x)
axes[1].set_xticklabels(
    B_error_summary["ws_bin"],
    rotation=45,
    ha="right",
)

for ax in axes:
    format_axes(ax)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_5_model_B_error_decomposition.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 5.6 turbine-power sensitivity to a ±0.5 m/s wind-speed change

scada = pd.read_parquet(
    SCADA_FILE
).copy()

scada["timestamp"] = pd.to_datetime(
    scada["timestamp"],
    utc=True,
)

training_end = pd.Timestamp(
    "2021-06-10",
    tz="UTC",
)

training_scada = scada[
    scada["timestamp"] < training_end
].copy()

curve_edges = np.arange(
    0,
    30.5,
    0.5,
)

curves = {}

for turbine_id in turbine_ids:
    turbine_data = training_scada[
        training_scada["turbine_id"]
        == turbine_id
    ].copy()

    turbine_data["ws_bin"] = pd.cut(
        turbine_data["wind_speed"],
        bins=curve_edges,
        include_lowest=True,
    )

    median_power = (
        turbine_data
        .groupby(
            "ws_bin",
            observed=True,
        )["power_kw"]
        .median()
        .dropna()
    )

    curves[turbine_id] = (
        np.array(
            [
                interval.mid
                for interval in median_power.index
            ]
        ),
        median_power.to_numpy(),
    )

wind_speed_grid = np.arange(
    4.0,
    18.01,
    0.1,
)

sensitivity_by_turbine = []

for turbine_id in turbine_ids:
    midpoints, power_values = curves[
        turbine_id
    ]

    power_plus = np.interp(
        wind_speed_grid + 0.5,
        midpoints,
        power_values,
    )

    power_minus = np.interp(
        wind_speed_grid - 0.5,
        midpoints,
        power_values,
    )

    sensitivity_by_turbine.append(
        np.abs(
            power_plus - power_minus
        ) / 2
    )

sensitivity_by_turbine = np.vstack(
    sensitivity_by_turbine
)

sensitivity_mean = (
    sensitivity_by_turbine.mean(axis=0)
)

sensitivity_sd = (
    sensitivity_by_turbine.std(
        axis=0,
        ddof=1,
    )
)

fig, ax = plt.subplots(
    figsize=(7.5, 5)
)

ax.plot(
    wind_speed_grid,
    sensitivity_mean,
    linewidth=2,
    color=COLOR_REFERENCE,
)

ax.fill_between(
    wind_speed_grid,
    sensitivity_mean - sensitivity_sd,
    sensitivity_mean + sensitivity_sd,
    alpha=0.18,
    color=COLOR_REFERENCE,
    label="±1 SD across turbines",
)

ax.set_xlabel(
    "Wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Mean turbine power change (kW)",
    fontsize=LABEL_FONTSIZE,
)

format_axes(ax)

ax.legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_6_power_curve_sensitivity.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# table 5.4 turbine-to-farm error cancellation

cancellation_rows = []

for seed in SEEDS:
    data = predictions[seed]

    actual_turbine_power = np.column_stack(
        [
            data[
                f"actual_power_t{turbine_id}"
            ].to_numpy()
            for turbine_id in turbine_ids
        ]
    )

    for model in ["B", "C"]:
        predicted_turbine_power = np.column_stack(
            [
                data[
                    f"pred_{model}_power_t{turbine_id}"
                ].to_numpy()
                for turbine_id in turbine_ids
            ]
        )

        gross_turbine_error = np.mean(
            np.sum(
                np.abs(
                    predicted_turbine_power
                    - actual_turbine_power
                ),
                axis=1,
            )
        )

        farm_error = np.mean(
            np.abs(
                predicted_turbine_power.sum(axis=1)
                - actual_turbine_power.sum(axis=1)
            )
        )

        cancellation_rows.append(
            {
                "seed": seed,
                "model": f"Model {model}",
                "cancellation_pct": (
                    100
                    * (
                        1
                        - farm_error
                        / gross_turbine_error
                    )
                ),
            }
        )

cancellation_by_seed = pd.DataFrame(
    cancellation_rows
)

table_5_4 = (
    cancellation_by_seed
    .groupby("model")
    .agg(
        mean=("cancellation_pct", "mean"),
        sd=("cancellation_pct", "std"),
    )
    .reset_index()
)

print(
    table_5_4
    .round(2)
    .to_string(index=False)
)


In [ ]:
# figure 5.7 turbine-level error under wake exposure

B_wake = pd.read_csv(
    WAKE_RESULTS_DIR
    / "model_B_wake_error_summary.csv"
)

C_wake = pd.read_csv(
    WAKE_RESULTS_DIR
    / "model_C_wake_error_summary.csv"
)

plot_labels = [
    "4–6",
    "6–8",
    "8–10",
    "10–12",
    "12–15",
]

wake_classes = [
    "Low",
    "Medium",
    "High",
]

x = np.arange(len(plot_labels))
width = 0.24

fig, axes = plt.subplots(
    2,
    1,
    figsize=(7.5, 7.2),
    sharex=True,
)

for class_index, wake_class in enumerate(
    wake_classes
):
    B_temp = (
        B_wake[
            B_wake["wake_class"]
            == wake_class
        ]
        .set_index("ws_bin")
        .reindex(plot_labels)
    )

    axes[0].bar(
        x
        + (class_index - 1)
        * width,
        B_temp["MAE_mean"],
        width,
        yerr=B_temp["MAE_sd"],
        capsize=3,
        label=wake_class,
        color=WAKE_COLORS[
            wake_class
        ],
    )

    C_temp = (
        C_wake[
            C_wake["wake_class"]
            == wake_class
        ]
        .set_index("ws_bin")
        .reindex(plot_labels)
    )

    axes[1].bar(
        x
        + (class_index - 1)
        * width,
        C_temp["MAE_mean"],
        width,
        yerr=C_temp["MAE_sd"],
        capsize=3,
        label=wake_class,
        color=WAKE_COLORS[
            wake_class
        ],
    )

axes[0].set_ylabel(
    "Model B wind-speed MAE (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_ylabel(
    "Model C turbine-power MAE (kW)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xlabel(
    "Global wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_labels)

for ax in axes:
    format_axes(
        ax,
        grid_axis="y",
    )

axes[0].legend(
    title="Wake severity",
    frameon=True,
    fontsize=LEGEND_FONTSIZE,
    title_fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_7_wake_exposure_error.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 5.8 observed and predicted relative farm power by wind direction

relative_power = pd.read_csv(
    WAKE_RESULTS_DIR
    / "directional_relative_power_summary.csv"
)

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

ax.plot(
    relative_power["dir_bin"],
    relative_power["Observed"],
    color=COLOR_OBSERVED,
    linewidth=2.2,
    label="Observed",
)

ax.plot(
    relative_power["dir_bin"],
    relative_power["A_mean"],
    color=COLOR_A,
    linewidth=1.6,
    label="Model A",
)

ax.plot(
    relative_power["dir_bin"],
    relative_power["B_mean"],
    color=COLOR_B,
    linewidth=1.6,
    label="Model B",
)

ax.plot(
    relative_power["dir_bin"],
    relative_power["C_mean"],
    color=COLOR_C,
    linewidth=1.6,
    linestyle="--",
    label="Model C",
)

ax.axhline(
    1,
    color="0.4",
    linestyle="--",
    linewidth=1,
    label="No-wake reference",
)

ax.set_xlabel(
    "Wind direction (°)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Relative farm power",
    fontsize=LABEL_FONTSIZE,
)

ax.set_xlim(0, 360)
ax.set_xticks(
    np.arange(0, 361, 30)
)

format_axes(ax)

ax.legend(
    ncol=3,
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_8_relative_power_direction.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 5.9 mean farm-power residual by wind direction

residuals = pd.read_csv(
    WAKE_RESULTS_DIR
    / "directional_residuals_summary.csv"
)

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

ax.plot(
    residuals["dir_bin"],
    residuals["A_mean"],
    color=COLOR_A,
    linewidth=1.6,
    label="Model A",
)

ax.plot(
    residuals["dir_bin"],
    residuals["B_mean"],
    color=COLOR_B,
    linewidth=1.6,
    label="Model B",
)

ax.plot(
    residuals["dir_bin"],
    residuals["C_mean"],
    color=COLOR_C,
    linewidth=1.6,
    linestyle="--",
    label="Model C",
)

ax.axhline(
    0,
    color="0.4",
    linestyle="--",
    linewidth=1,
)

ax.set_xlabel(
    "Wind direction (°)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Mean farm-power residual (kW)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_xlim(0, 360)
ax.set_xticks(
    np.arange(0, 361, 30)
)

format_axes(ax)

ax.legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "fig_5_9_directional_residuals.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# table 5.5 high-wake versus non-high-wake conditions

table_5_5 = pd.read_csv(
    WAKE_RESULTS_DIR
    / "high_vs_non_high_wake_summary.csv"
)

print(
    table_5_5
    .round(4)
    .to_string(index=False)
)


In [ ]:
# save the key result tables used in the dissertation

TABLE_DIR = (
    Path("results")
    / "final_tables"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

table_5_1.to_csv(
    TABLE_DIR / "table_5_1_model_performance.csv",
    index=False,
)

table_5_2.to_csv(
    TABLE_DIR / "table_5_2_capacity.csv",
    index=False,
)

table_5_3.to_csv(
    TABLE_DIR / "table_5_3_model_B_reference.csv",
    index=False,
)

table_5_4.to_csv(
    TABLE_DIR / "table_5_4_error_cancellation.csv",
    index=False,
)

table_5_5.to_csv(
    TABLE_DIR / "table_5_5_wake_comparison.csv",
    index=False,
)

spread.to_csv(
    TABLE_DIR / "figure_5_2_spread_values.csv",
    index=False,
)

B_error_summary.to_csv(
    TABLE_DIR / "figure_5_5_model_B_values.csv",
    index=False,
)

print(f"saved results figures to {FIGURE_DIR}")
print(f"saved result tables to {TABLE_DIR}")
